**Module:** 02 Mathematics  
**Part:** D — Optimization  
**Duration:** ~3 hours

---

## Learning Objectives

1. Derive momentum update rule
2. Derive RMSProp and Adam update rules
3. Implement Adam optimizer from scratch
4. Compare optimizers on same problem


## 1. Momentum

Accumulates velocity to smooth updates:
$$\mathbf{v}_{t+1} = \beta \mathbf{v}_t + \nabla L(\mathbf{w}_t)$$
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \mathbf{v}_{t+1}$$

Like a ball rolling downhill — accelerates in consistent directions.

## 2. Adam (Adaptive Moment Estimation)

Combines momentum (1st moment) and RMSProp (2nd moment):

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

Default: $\beta_1=0.9, \beta_2=0.999, \epsilon=10^{-8}$

Your **water-bodies-detection** uses AdamW (Adam + decoupled weight decay).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11
rng = np.random.default_rng(42)

In [ ]:
def adam_optimizer(grad_fn, w_init, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, steps=100):
    w = w_init.copy()
    m, v = np.zeros_like(w), np.zeros_like(w)
    history = []
    for t in range(1, steps + 1):
        g = grad_fn(w)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        w -= lr * m_hat / (np.sqrt(v_hat) + eps)
        history.append(w.copy())
    return w, history

# Minimize f(w) = w[0]**2 + 10*w[1]**2 (ill-conditioned)
grad = lambda w: np.array([2*w[0], 20*w[1]])
w0 = np.array([5.0, 5.0])
w_final, hist = adam_optimizer(grad, w0, lr=0.5, steps=50)
print(f'Start: {w0}, End: {w_final}')

In [ ]:
# Compare SGD, Momentum, Adam
w_sgd, w_mom, w_adam = [np.array([5., 5.]) for _ in range(3)]
v = np.zeros(2)
hist_sgd, hist_mom, hist_adam = [], [], []

for t in range(100):
    g = np.array([2*w_sgd[0], 20*w_sgd[1]])
    w_sgd -= 0.01 * g; hist_sgd.append(w_sgd.copy())
    
    g = np.array([2*w_mom[0], 20*w_mom[1]])
    v = 0.9 * v + g
    w_mom -= 0.01 * v; hist_mom.append(w_mom.copy())
    
    _, h = adam_optimizer(lambda w: np.array([2*w[0], 20*w[1]]), w_adam, lr=0.1, steps=1)
    w_adam = h[0]; hist_adam.append(w_adam.copy())

for hist, lbl in [(hist_sgd,'SGD'),(hist_mom,'Momentum'),(hist_adam,'Adam')]:
    h = np.array(hist)
    plt.plot(h[:,0], h[:,1], 'o-', markersize=2, label=lbl, alpha=0.7)
plt.plot(0, 0, 'k*', markersize=15, label='Optimum')
plt.xlabel('w0'); plt.ylabel('w1'); plt.legend(); plt.title('Optimizer Trajectories'); plt.show()

## Exercise 14

Implement RMSProp from scratch. Compare convergence with SGD on $f(w_0, w_1) = w_0^2 + 10w_1^2$.

In [ ]:
# YOUR CODE HERE


---

## Interview Questions

See module quiz and exercises.

## Summary

Adam is the default optimizer in deep learning. Understanding its update rule is essential.

**Notebook 20 complete.**

**Next:** [21_Regression_Losses.ipynb](21_Regression_Losses.ipynb)